In [1]:
import random
import time
from time import perf_counter
import math
import os
from bisect import bisect_left
from concurrent.futures import ThreadPoolExecutor
import heapq
import cupy as cp
from IPython.display import display, Markdown, HTML
from multiprocessing import Pool, cpu_count
import heapq
import math

os.environ['LD_LIBRARY_PATH'] = '/home/chaul/hpc_gpu/lib/python3.10/site-packages/nvidia/cu13/lib:' + os.environ.get('LD_LIBRARY_PATH', '')

In [2]:
def print_fmt(text, tamanho=12, negrito=False, italico=False, cor="black"):
    """
    Exibe texto formatado no Jupyter.
    
    Parâmetros:
      text    : str (pode ser f"...")
      tamanho : int (px)
      negrito : bool
      italico : bool
      cor     : str (ex: 'black', 'red', '#1f77b4')
    """
    weight = "bold" if negrito else "normal"
    style  = "italic" if italico else "normal"

    html = (
        f"<span style='"
        f"font-size:{tamanho}px;"
        f"font-weight:{weight};"
        f"font-style:{style};"
        f"color:{cor};"
        f"'>"
        f"{text}"
        f"</span>"
    )
    display(HTML(html))

In [3]:
# ============================================================
# Utilitários
# ============================================================
def is_sorted_non_decreasing(x):
    # x pode ser list/np.ndarray
    for i in range(1, len(x)):
        if x[i] < x[i-1]:
            return False
    return True

def cuda_warmup_context(device_id=0):
    cp.cuda.Device(device_id).use()
    _ = cp.sort(cp.random.random(200_000))
    cp.cuda.Stream.null.synchronize()

def make_base_array(n=10_000, seed=42):
    rng = random.Random(seed)
    return [rng.random() for _ in range(n)]  # floats em [0,1)

def is_sorted(a):
    return all(a[i] <= a[i+1] for i in range(len(a)-1))

def benchmark(fn, base, name):
    data = base[:]  # cópia para preservar o memo array inicial
    t0 = time.perf_counter()
    out = fn(data)
    t1 = time.perf_counter()

    # Algumas funções ordenam "in-place" e retornam a própria lista.
    # Outras retornam uma nova lista.
    result = out if out is not None else data

    ok = is_sorted(result)
    print(f"{name:28s} | ok={ok} | {t1 - t0:.4f}s")
    return result

# ============================================================
# (a) Técnica lenta: dois laços for (Selection Sort)
# O(n^2) e bem lenta para 10k — propositalmente.
# ============================================================

def selection_sort_two_loops(a):
    n = len(a)
    for i in range(n - 1):
        min_idx = i
        ai = a[i]
        for j in range(i + 1, n):
            if a[j] < a[min_idx]:
                min_idx = j
        if min_idx != i:
            a[i], a[min_idx] = a[min_idx], ai
    return a

# ============================================================
# (b) Usando busca binária: Insertion Sort com bisect
# Mantém lista ordenada e insere cada elemento via busca binária.
# Também O(n^2)
# ============================================================

def binary_insertion_sort(a):
    out = []
    for x in a:
        idx = bisect_left(out, x)  # busca binária para achar posição
        out.insert(idx, x)         # inserção em lista é O(n)
    return out

# ============================================================
# (c) Threads em CPU: divide em chunks, ordena em threads, e faz merge
# Observação: em CPython, o GIL limita ganho em CPU-bound.
# ============================================================

def _sort_chunk(chunk):
    return sorted(chunk)                 # nativo da biblioteca
    #return binary_insertion_sort(chunk)  # nossa implementação

def threaded_sort_merge(a, num_threads=None):
    n = len(a)
    if num_threads is None:
        num_threads = min(8, (os.cpu_count() or 4))

    # Particiona em blocos aproximadamente iguais
    chunk_size = math.ceil(n / num_threads)
    chunks = [a[i:i + chunk_size] for i in range(0, n, chunk_size)]

    with ThreadPoolExecutor(max_workers=num_threads) as ex:
        sorted_chunks = list(ex.map(_sort_chunk, chunks))

    # Merge k-way (heapq.merge) produz iterador ordenado
    merged_iter = heapq.merge(*sorted_chunks)
    return list(merged_iter)


# ============================================================
# (d) Threads em CPU, com paralelismo
# divide em chunks, ordena em threads, e faz merge
# Requer: CUDA + driver + cupy instalado.
# ============================================================    

def mp_sort_merge(a, num_proc=None):
    if num_proc is None:
        num_proc = min(8, cpu_count())

    n = len(a)
    chunk_size = math.ceil(n / num_proc)
    chunks = [a[i:i+chunk_size] for i in range(0, n, chunk_size)]

    with Pool(num_proc) as p:
        sorted_chunks = p.map(sorted, chunks)

    return list(heapq.merge(*sorted_chunks)) 

# ============================================================
# (e) GPU e computação paralela: CuPy (cp.sort)
# Requer: CUDA + driver + cupy instalado.
# ============================================================
def gpu_sort_cupy(a):
    x = cp.asarray(a)      # host -> device
    y = cp.sort(x)         # sort na GPU
    return cp.asnumpy(y).tolist()  # device -> host

In [4]:
def mp_sort_merge_timed(a, num_proc=None, start_method_hint=None):
    """
    Retorna:
      out (list) e um dict com tempos:
      - t_chunk: particionamento
      - t_map: Pool.map (inclui IPC + sort em cada processo)
      - t_merge: heapq.merge + materialização list()
      - t_total: total
      - k: número de chunks
    """
    if num_proc is None:
        num_proc = min(8, cpu_count())

    # garante list/np array indexável
    n = len(a)

    t0 = perf_counter()

    # chunking
    t1 = perf_counter()
    chunk_size = math.ceil(n / num_proc)
    chunks = [a[i:i + chunk_size] for i in range(0, n, chunk_size)]
    t2 = perf_counter()

    # map (IPC + sort local)
    with Pool(processes=num_proc) as p:
        sorted_chunks = p.map(sorted, chunks)
    t3 = perf_counter()

    # merge (sequencial)
    merged_iter = heapq.merge(*sorted_chunks)
    out = list(merged_iter)
    t4 = perf_counter()

    times = {
        "t_chunk": t2 - t1,
        "t_map":   t3 - t2,
        "t_merge": t4 - t3,
        "t_total": t4 - t0,
        "k": len(sorted_chunks),
        "n": n,
        "num_proc": num_proc,
    }
    return out, times   
    
def benchmark_timed_mp(base, num_proc, label):
    out, tt = mp_sort_merge_timed(base, num_proc=num_proc)
    ok = (len(out) == len(base)) and is_sorted_non_decreasing(out)

    print(
        f"{label:<38} | ok={ok} | "
        f"chunk={tt['t_chunk']:.4f}s | map(IPC+sort)={tt['t_map']:.4f}s | "
        f"merge={tt['t_merge']:.4f}s | total={tt['t_total']:.4f}s | "
        f"k={tt['k']}"
    )
    return out, tt    

In [5]:
# ============================================================
# Execução
# ============================================================
base = make_base_array(n=10_000, seed=42)  # memo array inicial


In [6]:
print_fmt(f"Total de elementos: {len(base):_}".replace("_", "."), tamanho=22, negrito=True)
print('-'*100)
# (a) lento (dois for)
benchmark(selection_sort_two_loops, base, "(a) two loops / selection")
print('-'*100)
# (b) busca binária (bisect)
benchmark(binary_insertion_sort, base,                                "(b) binary insertion      ")
print('-'*100)
# (c) threads CPU
benchmark(lambda arr: threaded_sort_merge(arr, num_threads=4), base,  "(c)  4 threads chunk+merge");
benchmark(lambda arr: threaded_sort_merge(arr, num_threads=8), base,  "(c)  8 threads chunk+merge");
benchmark(lambda arr: threaded_sort_merge(arr, num_threads=12), base, "(c) 12 threads chunk+merge");    
benchmark(lambda arr: threaded_sort_merge(arr, num_threads=24), base, "(c) 24 threads chunk+merge");        
benchmark(lambda arr: threaded_sort_merge(arr, num_threads=48), base, "(c) 48 threads chunk+merge");    
print('-'*100)
# (d) threads CPU com paralelismo
benchmark(lambda arr: mp_sort_merge(arr, num_proc=4), base,  "(d)  4 processadores chunk+merge");
benchmark(lambda arr: mp_sort_merge(arr, num_proc=8), base,  "(d)  8 processadores chunk+merge");
benchmark(lambda arr: mp_sort_merge(arr, num_proc=12), base, "(d) 12 processadores chunk+merge");    
benchmark(lambda arr: mp_sort_merge(arr, num_proc=16), base, "(d) 16 processadores chunk+merge");   
print('-'*100)
#cuda_warmup_context()
benchmark(gpu_sort_cupy, base, "GPU 1ª execução sort (CuPy)");
benchmark(gpu_sort_cupy, base, "GPU 2ª execução sort (CuPy)");

----------------------------------------------------------------------------------------------------
(a) two loops / selection    | ok=True | 1.4629s
----------------------------------------------------------------------------------------------------
(b) binary insertion         | ok=True | 0.0072s
----------------------------------------------------------------------------------------------------
(c)  4 threads chunk+merge   | ok=True | 0.0033s
(c)  8 threads chunk+merge   | ok=True | 0.0033s
(c) 12 threads chunk+merge   | ok=True | 0.0039s
(c) 24 threads chunk+merge   | ok=True | 0.0042s
(c) 48 threads chunk+merge   | ok=True | 0.0051s
----------------------------------------------------------------------------------------------------
(d)  4 processadores chunk+merge | ok=True | 0.0442s
(d)  8 processadores chunk+merge | ok=True | 0.0460s
(d) 12 processadores chunk+merge | ok=True | 0.0658s
(d) 16 processadores chunk+merge | ok=True | 0.0763s
-----------------------------------------

# 1. A Lei de Amdahl

## A aceleração máxima (speedup) de um programa paralelo é limitada pela parte sequencial do código.

In [7]:
base = make_base_array(n=20_000_000, seed=42)
print_fmt(f"Total de elementos: {len(base):_}".replace("_", "."), tamanho=22, negrito=True)
_ = benchmark_timed_mp(base, 4,  "(d)  4 processos chunk+merge")
_ = benchmark_timed_mp(base, 8,  "(d)  8 processos chunk+merge")
_ = benchmark_timed_mp(base, 12, "(d) 12 processos chunk+merge")
_ = benchmark_timed_mp(base, 16, "(d) 16 processos chunk+merge")
print('-'*130)
benchmark(gpu_sort_cupy, base, "GPU execução sort (CuPy)");
print('-'*130)
print_fmt("IPC: Inter-Process Communication (Comunicação entre Processos)", tamanho=11, italico=True, cor="gray")

(d)  4 processos chunk+merge           | ok=True | chunk=0.1798s | map(IPC+sort)=3.5301s | merge=2.1721s | total=5.8820s | k=4
(d)  8 processos chunk+merge           | ok=True | chunk=0.0942s | map(IPC+sort)=2.3463s | merge=2.6405s | total=5.0809s | k=8
(d) 12 processos chunk+merge           | ok=True | chunk=0.0893s | map(IPC+sort)=2.2027s | merge=3.0435s | total=5.3355s | k=12
(d) 16 processos chunk+merge           | ok=True | chunk=0.0931s | map(IPC+sort)=2.4123s | merge=3.6323s | total=6.1378s | k=16
----------------------------------------------------------------------------------------------------------------------------------
GPU execução sort (CuPy)     | ok=True | 1.3158s
----------------------------------------------------------------------------------------------------------------------------------
